# 신호등 YOLOv8n 재학습 (Colab GPU)
Go2 신호등 5클래스(tl_red, tl_green, pedestrian, temi, unknown). 데이터: `ds_aug.zip` (학습 1,242장 / 검증 103장).

**시작 전**: 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 선택. 그다음 셀을 위에서부터 차례로 실행.

결과는 구글 드라이브 `MyDrive/tl5_runs/` 에 저장되므로 연결이 끊겨도 남는다. 끊기면 1~5번 셀을 다시 실행하면 6번이 이어서 학습한다.

In [ ]:
# 1. GPU 확인 — 'Tesla T4' 같은 이름이 나와야 한다
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. ultralytics 설치
!pip -q install ultralytics
import ultralytics; ultralytics.checks()

In [ ]:
# 3. 구글 드라이브 연결 (팝업에서 ds_aug.zip 이 있는 계정으로 허용)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. 데이터셋을 Colab 로컬 디스크로 복사·압축 해제 (드라이브에서 바로 읽으면 느리다)
import os, glob, subprocess
ZIP = '/content/drive/MyDrive/ds_aug.zip'   # ← 다른 폴더에 있으면 여기만 고친다
if not os.path.exists(ZIP):
    print('없음:', ZIP); print('후보:', glob.glob('/content/drive/MyDrive/**/ds_aug*.zip', recursive=True)[:5])
else:
    !cp "$ZIP" /content/ds_aug.zip && unzip -q -o /content/ds_aug.zip -d /content/
    print('풀림:', os.listdir('/content/ds_aug'))

In [ ]:
# 5. data.yaml 경로를 절대경로로 고치고 개수 확인 (상대경로 그대로면 'images not found' 로 실패)
!sed -i "s|^path: .*|path: /content/ds_aug|" /content/ds_aug/data.yaml
!cat /content/ds_aug/data.yaml
import glob
for sp, want in (('train', 1242), ('val', 103)):
    n = len(glob.glob(f'/content/ds_aug/images/{sp}/*.jpg')); m = len(glob.glob(f'/content/ds_aug/labels/{sp}/*.txt'))
    print(sp, '이미지', n, '라벨', m, '✓' if n == m == want else f'★ 예상 {want}')

In [ ]:
# 6. 학습 — 1280 해상도, 최대 100에폭, 20에폭 동안 안 나아지면 조기 종료. T4 기준 대략 1~1.5시간
#    결과는 드라이브에 바로 저장되고, 끊긴 뒤 다시 실행하면 last.pt 에서 이어서 한다
import os
from ultralytics import YOLO
PROJ, NAME = '/content/drive/MyDrive/tl5_runs', 'tl5_1280_aug_retrain'
last = f'{PROJ}/{NAME}/weights/last.pt'
if os.path.exists(last):
    print('이어서 학습:', last); YOLO(last).train(resume=True)
else:
    YOLO('yolov8n.pt').train(data='/content/ds_aug/data.yaml', imgsz=1280, epochs=100, patience=20, batch=16,
                             project=PROJ, name=NAME, exist_ok=True)

In [ ]:
# 7. 클래스별 검증 — 9월 기준·NUC 예비 모델과 비교 (가장 중요한 건 tl_green 의 P)
from ultralytics import YOLO
best = '/content/drive/MyDrive/tl5_runs/tl5_1280_aug_retrain/weights/best.pt'
m = YOLO(best); r = m.val(data='/content/ds_aug/data.yaml', imgsz=1280, plots=True); b = r.box
print(f"{'':10s} {'P':>6s} {'R':>6s} {'mAP50':>6s} {'50-95':>6s}")
print(f"{'전체':10s} {b.mp:6.3f} {b.mr:6.3f} {b.map50:6.3f} {b.map:6.3f}")
for i, n in m.names.items(): print(f'{n:10s} {b.p[i]:6.3f} {b.r[i]:6.3f} {b.ap50[i]:6.3f} {b.ap[i]:6.3f}')
print('\n비교 기준 (같은 검증셋 103장·348박스)')
print('9월 1280+증강   전체 P 0.951 R 0.906 mAP50 0.945 mAP50-95 0.681 | tl_green P 0.981 R 0.895')
print('NUC 640 예비    전체 P 0.927 R 0.890 mAP50 0.941 mAP50-95 0.677 | tl_green P 0.876 R 0.789')

In [ ]:
# 8. ONNX 변환 + 결과 정리 (내부 PC TensorRT 8.5 호환: opset 12, 고정 크기)
import shutil, os
from ultralytics import YOLO
d = '/content/drive/MyDrive/tl5_runs/tl5_1280_aug_retrain'
onnx = YOLO(f'{d}/weights/best.pt').export(format='onnx', imgsz=1280, opset=12, simplify=True, dynamic=False)
out = '/content/drive/MyDrive/tl5_result'; os.makedirs(out, exist_ok=True)
shutil.copy(f'{d}/weights/best.pt', f'{out}/tl5_1280_aug_retrain.pt')
shutil.copy(onnx, f'{out}/tl5_1280_aug_retrain.onnx')
for f in ('results.csv', 'args.yaml', 'confusion_matrix.png', 'results.png'):
    if os.path.exists(f'{d}/{f}'): shutil.copy(f'{d}/{f}', out)
print('드라이브 MyDrive/tl5_result 에 저장:', os.listdir(out))

## 끝나면
`MyDrive/tl5_result/` 폴더(가중치 .pt, .onnx, results.csv 등)를 받아서 NUC `~/Downloads/` 에 넣는다.
이후 NUC 에서 같은 검증셋으로 다시 확인하고, 로봇 충전 후 내부 PC(Orin NX)에서 TensorRT INT8 로 변환한다.